In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time 

In [23]:
#urls
base_url = "https://www.federalreserve.gov"
hist_url = "https://www.federalreserve.gov/monetarypolicy/materials/assets/final-hist.json"
recent_url= "https://www.federalreserve.gov/monetarypolicy/materials/assets/final-recent.json"

In [ ]:
hist_data = requests.get(hist_url).json()
print(hist_data['mtgitems'][0])

{'d': '2010-01-27', 'mtg': 'January 27, 2010', 'type': 'Ag', 'name': 'Agenda', 'url': '/monetarypolicy/files/FOMC20100127Agenda.pdf', 'size': 14.86, 'dt': '2010-01-27'}


In [26]:
recent_data=requests.get(recent_url).json()
print(recent_data['mtgitems'][0])

{'d': '2025-08-22', 'mtg': 'August 22, 2025', 'type': 'Pl', 'name': 'Statement on Longer-Run Goals and Monetary Policy Strategy', 'url': '/newsevents/pressreleases/monetary20250822a.htm', 'dt': '2025-08-22'}


In [ ]:
#list
hist_statements = [item for item in hist_data['mtgitems'] 
              if item['type'] == 'St' 
              and item['d'] >= '2008-01-01']

In [29]:
recent_statements= [item for item in recent_data['mtgitems']
                    if item['type']=='St']

In [30]:
print(hist_statements[0])

{'d': '2010-01-27', 'mtg': 'January 27, 2010', 'type': 'St', 'name': 'Statement', 'url': '/newsevents/press/monetary/20100127a.htm', 'dt': '2010-01-27'}


In [31]:
print(recent_statements[0])

{'d': '2025-09-17', 'mtg': 'September 16-17, 2025', 'type': 'St', 'dt': '2025-09-17', 'files': [{'name': 'HTML', 'url': '/newsevents/pressreleases/monetary20250917a.htm'}, {'name': 'PDF', 'url': '/monetarypolicy/files/monetary20250917a1.pdf'}]}


In [32]:
statements=hist_statements+recent_statements

In [33]:
print(f"Historical statemnts: {len(hist_statements)}")
print(f"Recent Statements: {len(recent_statements)}")
print(f"Total statements: {len(statements)}")

Historical statemnts: 28
Recent Statements: 124
Total statements: 152


In [38]:
results = []

for i, statement in enumerate(statements):
    # Handle both URL structures
    if 'url' in statement:
        full_url = base_url + statement['url']
    elif 'files' in statement:
        full_url = base_url + statement['files'][0]['url']
    else:
        continue
    
    try:
        response = requests.get(full_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        content_div = soup.find('div', class_='col-xs-12 col-sm-8 col-md-8')
        text = content_div.get_text(strip=True) if content_div else None
        
        results.append({
            'date': statement['d'],
            'meeting': statement['mtg'],
            'url': full_url,
            'text': text
        })
        
        print(f"[{i+1}/{len(statements)}] Scraped: {statement['d']}")
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error on {statement['d']}: {e}")

df = pd.DataFrame(results)
df.to_csv('data/raw/fomc_statements.csv', index=False)
print(f"\nSaved {len(df)} statements")

[1/152] Scraped: 2010-01-27
[2/152] Scraped: 2010-03-16
[3/152] Scraped: 2010-04-28
[4/152] Scraped: 2010-05-09
[5/152] Scraped: 2010-06-23
[6/152] Scraped: 2010-08-10
[7/152] Scraped: 2010-09-21
[8/152] Scraped: 2010-11-03
[9/152] Scraped: 2010-12-14
[10/152] Scraped: 2009-01-28
[11/152] Scraped: 2009-03-17
[12/152] Scraped: 2009-04-29
[13/152] Scraped: 2009-06-24
[14/152] Scraped: 2009-08-11
[15/152] Scraped: 2009-09-22
[16/152] Scraped: 2009-11-04
[17/152] Scraped: 2009-12-15
[18/152] Scraped: 2008-01-21
[19/152] Scraped: 2008-01-30
[20/152] Scraped: 2008-03-10
[21/152] Scraped: 2008-03-18
[22/152] Scraped: 2008-04-30
[23/152] Scraped: 2008-06-25
[24/152] Scraped: 2008-08-08
[25/152] Scraped: 2008-09-16
[26/152] Scraped: 2008-10-07
[27/152] Scraped: 2008-10-29
[28/152] Scraped: 2008-12-16
[29/152] Scraped: 2025-09-17
[30/152] Scraped: 2025-10-29
[31/152] Scraped: 2025-12-10
[32/152] Scraped: 2026-01-28
[33/152] Scraped: 2025-05-07
[34/152] Scraped: 2025-06-18
[35/152] Scraped: 2025-